# Notebook 02: Feature Stores

## Why This Matters

Feature stores emerged as a critical ML infrastructure component after companies like Uber (Michelangelo, 2017), Airbnb (Zipline, 2018), and LinkedIn (Feathr) independently built similar internal systems to solve the same problem: feature duplication and training-serving skew. Today, feature stores appear in virtually every ML system design interview involving real-time prediction, because they represent the boundary between data engineering and ML engineering. Open-source tools like Feast and commercial offerings like Tecton and Hopsworks have made this pattern mainstream, and understanding their architecture—especially the online/offline split—is expected knowledge at senior ML engineer levels.

## 1. What Problem Does a Feature Store Solve?

Without a feature store, teams reimplement the same feature logic in Python for training, in Java/Go for serving, and in SQL for analytics. This creates training-serving skew—where the model sees different data at training vs prediction time—which is one of the most insidious silent failures in production ML. Feature stores solve this by providing a single source of truth for features with both batch (offline) and real-time (online) access patterns.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def box(ax, x, y, w, h, text, color, fontsize=9, text_color='white'):
    r = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color)

def arr(ax, x1, y1, x2, y2, color='gray'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))

# WITHOUT feature store
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('WITHOUT Feature Store', fontsize=12, fontweight='bold', color='#E74C3C')

box(ax, 3, 7.5, 4, 1, 'Raw Data Sources', '#7F8C8D')
box(ax, 0.5, 5, 3, 1.2, 'Python Feature\nLogic (Training)', '#E74C3C')
box(ax, 4, 5, 3, 1.2, 'Java Feature\nLogic (Serving)', '#E67E22')
box(ax, 7.5, 5, 2, 1.2, 'SQL Feature\nLogic (BI)', '#F39C12')
box(ax, 1, 2.8, 2.5, 1, 'Training\nDataset', '#2980B9')
box(ax, 4.2, 2.8, 2.5, 1, 'Prod\nPredictions', '#27AE60')

arr(ax, 5, 7.5, 2, 6.2)
arr(ax, 5, 7.5, 5.5, 6.2)
arr(ax, 5, 7.5, 8.5, 6.2)
arr(ax, 2, 5, 2.2, 3.8)
arr(ax, 5.5, 5, 5.5, 3.8)

ax.text(5, 1.5, '⚠ Training-Serving Skew!\nDuplicate logic → drift → silent bugs',
        ha='center', va='center', fontsize=9, color='#E74C3C', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#FADBD8', alpha=0.8))

# WITH feature store
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 9)
ax2.axis('off')
ax2.set_title('WITH Feature Store', fontsize=12, fontweight='bold', color='#27AE60')

box(ax2, 3, 7.5, 4, 1, 'Raw Data Sources', '#7F8C8D')

# Feature store (central)
r = FancyBboxPatch((2, 4.5), 6, 2.2, boxstyle="round,pad=0.2",
                   facecolor='#1A5276', edgecolor='#2ECC71', linewidth=3, alpha=0.92)
ax2.add_patch(r)
ax2.text(5, 6.0, 'FEATURE STORE', ha='center', va='center',
         fontsize=11, fontweight='bold', color='white')
ax2.text(5, 5.5, 'Single Feature Definition', ha='center', va='center',
         fontsize=9, color='#AED6F1')

box(ax2, 2.2, 4.7, 2.2, 0.7, 'Offline Store\n(S3/DWH)', '#2471A3', fontsize=8)
box(ax2, 5.6, 4.7, 2.2, 0.7, 'Online Store\n(Redis/DynamoDB)', '#1A8754', fontsize=8)

box(ax2, 0.5, 2.2, 3, 1, 'Training\nDataset', '#2980B9')
box(ax2, 4, 2.2, 2.5, 1, 'Batch\nPredictions', '#8E44AD')
box(ax2, 7, 2.2, 2.5, 1, 'Online\nPredictions', '#27AE60')

arr(ax2, 5, 7.5, 5, 6.7, '#2ECC71')
arr(ax2, 3.3, 4.5, 2, 3.2, '#2ECC71')
arr(ax2, 5, 4.5, 5.2, 3.2, '#2ECC71')
arr(ax2, 6.7, 4.5, 8.2, 3.2, '#2ECC71')

ax2.text(5, 1.2, '✓ One definition, consistent everywhere',
         ha='center', va='center', fontsize=9, color='#27AE60', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#D5F5E3', alpha=0.8))

plt.tight_layout()
plt.savefig('feature_store_motivation.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Feature Store Architecture: Online vs Offline

The defining characteristic of a feature store is its dual-store architecture. The offline store serves training jobs with historical point-in-time correct features. The online store serves prediction requests with low-latency lookups. The challenge is keeping both in sync without introducing skew.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('Feature Store Dual-Store Architecture', fontsize=14, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=8.5, text_color='white'):
    r = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
                       facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=text_color,
            multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#555'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.1, my, label, fontsize=7.5, color=color)

# Data sources (top)
box(ax, 0.5, 7.5, 3, 1, 'Batch Data\n(S3/GCS/Hive)', '#7F8C8D')
box(ax, 4.5, 7.5, 3, 1, 'Streaming Data\n(Kafka/Kinesis)', '#E67E22')
box(ax, 8.5, 7.5, 3, 1, 'Request-Time Data\n(user context)', '#8E44AD')

# Feature computation
box(ax, 1, 5.6, 2.5, 1.2, 'Batch\nComputation\n(Spark/dbt)', '#2980B9')
box(ax, 4.8, 5.6, 2.5, 1.2, 'Stream\nComputation\n(Flink)', '#E67E22')
box(ax, 8.5, 5.6, 3, 1.2, 'On-demand\nComputation\n(Lambda fns)', '#8E44AD')

# Stores
ax.add_patch(FancyBboxPatch((0.3, 2.8), 5.5, 2, boxstyle="round,pad=0.2",
                            facecolor='#1A5276', edgecolor='#3498DB', linewidth=2.5, alpha=0.9))
ax.text(3.05, 4.5, 'OFFLINE STORE', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax.text(3.05, 4.1, 'Parquet / Delta Lake / Iceberg', ha='center', va='center', fontsize=8, color='#AED6F1')
ax.text(3.05, 3.7, 'Point-in-time correct joins', ha='center', va='center', fontsize=8, color='#AED6F1')
ax.text(3.05, 3.3, 'For training dataset generation', ha='center', va='center', fontsize=8, color='#AED6F1')
ax.text(3.05, 2.95, 'Query latency: seconds-to-minutes', ha='center', va='center', fontsize=7.5, color='#85C1E9', style='italic')

ax.add_patch(FancyBboxPatch((7.2, 2.8), 5.5, 2, boxstyle="round,pad=0.2",
                            facecolor='#1A5276', edgecolor='#2ECC71', linewidth=2.5, alpha=0.9))
ax.text(9.95, 4.5, 'ONLINE STORE', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
ax.text(9.95, 4.1, 'Redis / DynamoDB / Cassandra', ha='center', va='center', fontsize=8, color='#A9DFBF')
ax.text(9.95, 3.7, 'Key-value lookup by entity ID', ha='center', va='center', fontsize=8, color='#A9DFBF')
ax.text(9.95, 3.3, 'Latest feature values only', ha='center', va='center', fontsize=8, color='#A9DFBF')
ax.text(9.95, 2.95, 'Query latency: < 10ms p99', ha='center', va='center', fontsize=7.5, color='#82E0AA', style='italic')

# Consumers (bottom)
box(ax, 0.5, 0.5, 3.5, 1.5, 'Training Pipeline\n(historical features\nwith labels)', '#6C3483')
box(ax, 5, 0.5, 2, 1.5, 'Batch\nScoring', '#2C3E50')
box(ax, 8, 0.5, 2, 1.5, 'Online\nInference\nService', '#145A32')
box(ax, 11, 0.5, 3.5, 1.5, 'Feature\nMonitoring &\nDrift Detection', '#6E2F01')

# Arrows
arr(ax, 2, 7.5, 2.2, 6.8, '', '#555')
arr(ax, 6, 7.5, 6, 6.8, '', '#E67E22')
arr(ax, 10, 7.5, 10, 6.8, '', '#8E44AD')
arr(ax, 2.2, 5.6, 3, 4.8, '', '#2980B9')
arr(ax, 6, 5.6, 4, 4.8, '', '#E67E22')
arr(ax, 10, 5.6, 10.5, 4.8, '', '#8E44AD')
arr(ax, 5.5, 3.8, 7.5, 3.8, 'materialization', '#27AE60')
arr(ax, 3, 2.8, 2.2, 2.0, '', '#6C3483')
arr(ax, 3, 2.8, 6, 2.0, '', '#2C3E50')
arr(ax, 10, 2.8, 9, 2.0, '', '#145A32')
arr(ax, 10, 2.8, 12, 2.0, '', '#6E2F01')

plt.tight_layout()
plt.savefig('feature_store_architecture.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Point-in-Time Correct Feature Joins

This is the hardest technical concept in feature stores. When generating training data, you must retrieve the feature values that were available *at the time the label was generated*, not the current values. Using future information creates data leakage. This is called a point-in-time (PIT) join or time-travel join.

In [ ]:
import pandas as pd
import numpy as np

# Simulate the point-in-time join problem

# Feature history: user's running average watch time (updated daily)
feature_history = pd.DataFrame([
    {'user_id': 'u1', 'timestamp': pd.Timestamp('2024-01-01'), 'avg_watch_time': 120},
    {'user_id': 'u1', 'timestamp': pd.Timestamp('2024-01-05'), 'avg_watch_time': 145},
    {'user_id': 'u1', 'timestamp': pd.Timestamp('2024-01-10'), 'avg_watch_time': 160},
    {'user_id': 'u1', 'timestamp': pd.Timestamp('2024-01-15'), 'avg_watch_time': 155},
    {'user_id': 'u2', 'timestamp': pd.Timestamp('2024-01-01'), 'avg_watch_time': 80},
    {'user_id': 'u2', 'timestamp': pd.Timestamp('2024-01-08'), 'avg_watch_time': 95},
    {'user_id': 'u2', 'timestamp': pd.Timestamp('2024-01-14'), 'avg_watch_time': 105},
])

# Training labels: events when predictions were made
label_events = pd.DataFrame([
    {'user_id': 'u1', 'event_time': pd.Timestamp('2024-01-07'), 'clicked': 1},
    {'user_id': 'u1', 'event_time': pd.Timestamp('2024-01-12'), 'clicked': 0},
    {'user_id': 'u2', 'event_time': pd.Timestamp('2024-01-09'), 'clicked': 1},
    {'user_id': 'u2', 'event_time': pd.Timestamp('2024-01-16'), 'clicked': 0},
])

def point_in_time_join(labels_df, features_df, feature_col):
    """For each label event, find the most recent feature value available at event_time."""
    results = []
    for _, label_row in labels_df.iterrows():
        user = label_row['user_id']
        event_time = label_row['event_time']
        
        # Features available BEFORE (not at) event time — strict less-than to avoid leakage
        available = features_df[
            (features_df['user_id'] == user) &
            (features_df['timestamp'] < event_time)  # strictly before
        ]
        
        if len(available) > 0:
            # Most recent available feature
            feature_val = available.sort_values('timestamp').iloc[-1][feature_col]
        else:
            feature_val = None  # cold start — no history
        
        results.append({
            **label_row.to_dict(),
            f'{feature_col}_at_event_time': feature_val
        })
    return pd.DataFrame(results)

# Correct PIT join
pit_result = point_in_time_join(label_events, feature_history, 'avg_watch_time')

# Wrong join: uses CURRENT (latest) feature value — causes leakage
latest_features = feature_history.sort_values('timestamp').groupby('user_id').last().reset_index()
wrong_result = label_events.merge(latest_features[['user_id', 'avg_watch_time']], on='user_id')
wrong_result = wrong_result.rename(columns={'avg_watch_time': 'avg_watch_time_WRONG_LEAKAGE'})

print("=" * 65)
print("CORRECT: Point-in-Time Join (uses feature value available at event time)")
print("=" * 65)
print(pit_result.to_string(index=False))

print("\n" + "=" * 65)
print("WRONG: Latest-value join (uses future information — DATA LEAKAGE)")
print("=" * 65)
print(wrong_result[['user_id', 'event_time', 'clicked', 'avg_watch_time_WRONG_LEAKAGE']].to_string(index=False))

print("\nDifference (future information leaked into training):")
for i, row in pit_result.iterrows():
    user = row['user_id']
    wrong_val = wrong_result[wrong_result['user_id'] == user].iloc[0]['avg_watch_time_WRONG_LEAKAGE']
    correct_val = row['avg_watch_time_at_event_time']
    leaked = wrong_val != correct_val
    print(f"  {user} at {row['event_time'].date()}: correct={correct_val}, wrong={wrong_val}, leaked={leaked}")

## 4. Feature Types and Their Characteristics

Different features have different freshness requirements and computation costs. Understanding when to use batch vs streaming vs on-demand computation is central to feature store design.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

feature_types = [
    {
        'name': 'Static / Slow Features',
        'examples': ['User demographics', 'Item category', 'Content creator tier'],
        'update_freq': 'Days to weeks',
        'computation': 'Batch (Spark)',
        'storage': 'Offline only (or cache in online)',
        'freshness_hours': 168,  # 1 week
        'cost': 1,
        'color': '#2980B9'
    },
    {
        'name': 'Aggregate / Historical Features',
        'examples': ['7-day CTR', '30-day spend', 'avg session length'],
        'update_freq': 'Hours to days',
        'computation': 'Batch (Spark/dbt)',
        'storage': 'Offline + Online (Redis)',
        'freshness_hours': 24,
        'cost': 2,
        'color': '#27AE60'
    },
    {
        'name': 'Real-time Aggregate Features',
        'examples': ['Last-1hr clicks', 'Trending score', 'Real-time inventory'],
        'update_freq': 'Minutes',
        'computation': 'Streaming (Flink)',
        'storage': 'Online only (Redis)',
        'freshness_hours': 0.17,  # 10 min
        'cost': 4,
        'color': '#E67E22'
    },
    {
        'name': 'Request-time / On-demand Features',
        'examples': ['Query-item similarity', 'User context at request', 'Price at lookup time'],
        'update_freq': 'Per request',
        'computation': 'Lambda / microservice',
        'storage': 'Not stored (computed live)',
        'freshness_hours': 0,
        'cost': 5,
        'color': '#E74C3C'
    },
]

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Table / info display
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Feature Type Taxonomy', fontsize=12, fontweight='bold')

headers = ['Feature Type', 'Update Freq', 'Computation', 'Freshness']
col_x = [0.1, 3.5, 5.5, 8.0]

for j, h in enumerate(headers):
    ax.text(col_x[j], 9.5, h, fontsize=9, fontweight='bold', color='#333')

for i, ft in enumerate(feature_types):
    y = 7.5 - i * 2.0
    r = FancyBboxPatch((0, y - 0.5), 9.8, 1.6, boxstyle="round,pad=0.1",
                       facecolor=ft['color'], alpha=0.15, edgecolor=ft['color'], linewidth=1)
    ax.add_patch(r)
    ax.text(col_x[0], y + 0.4, ft['name'], fontsize=8, fontweight='bold', color=ft['color'])
    for ex in ft['examples'][:2]:
        ax.text(col_x[0] + 0.2, y + 0.4 - ft['examples'][:2].index(ex) * 0.45,
                f'• {ex}', fontsize=7.5, color='#555')
    ax.text(col_x[1], y + 0.1, ft['update_freq'], fontsize=8, color='#333')
    ax.text(col_x[2], y + 0.1, ft['computation'], fontsize=8, color='#333')
    fresh_str = 'Real-time' if ft['freshness_hours'] == 0 else (
        f"{ft['freshness_hours']:.0f} min" if ft['freshness_hours'] < 1 else
        f"{ft['freshness_hours']:.0f} hr"
    )
    ax.text(col_x[3], y + 0.1, fresh_str, fontsize=8, color='#333')

# Scatter: cost vs freshness
ax2 = axes[1]
for ft in feature_types:
    fresh = ft['freshness_hours'] if ft['freshness_hours'] > 0 else 0.01
    ax2.scatter(fresh, ft['cost'], c=ft['color'], s=200, zorder=5, edgecolors='white', linewidth=2)
    ax2.annotate(ft['name'].split('/')[0].strip(), (fresh, ft['cost']),
                 xytext=(8, 8), textcoords='offset points', fontsize=8)

ax2.set_xscale('log')
ax2.set_xlabel('Feature Staleness (hours, log scale)', fontsize=10)
ax2.set_ylabel('Infrastructure Cost (relative)', fontsize=10)
ax2.set_title('Freshness vs Infrastructure Cost\n(fresher = more expensive)', fontsize=11, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 7)

# Tradeoff arrow
ax2.annotate('', xy=(0.02, 5.5), xytext=(200, 1.5),
             arrowprops=dict(arrowstyle='<->', color='gray', lw=2))
ax2.text(1, 4, 'freshness-cost\ntradeoff', fontsize=9, color='gray', rotation=25)

plt.tight_layout()
plt.savefig('feature_types.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Feature Store Interview Trade-offs

Interviewers test whether you understand when NOT to use a feature store, and what the main operational challenges are. Knowing the failure modes is as important as knowing the happy path.

In [ ]:
tradeoffs = {
    "Build vs Buy": {
        "Build (Airbnb Zipline, Uber Michelangelo)": [
            "Pros: Tightly integrated with internal data systems",
            "Pros: Custom SLAs and latency requirements met exactly",
            "Cons: 6-18 months engineering investment",
            "Cons: Ongoing maintenance burden",
            "When: >50 ML models, large ML org, existing data platform",
        ],
        "Open Source (Feast)": [
            "Pros: Free, growing community, cloud-agnostic",
            "Pros: Good offline store and basic online store",
            "Cons: Limited streaming support, manual materialization",
            "Cons: No enterprise support SLA",
            "When: Startup or small ML team, standard use cases",
        ],
        "Commercial (Tecton, Hopsworks)": [
            "Pros: Streaming support, monitoring, enterprise features",
            "Pros: Managed infrastructure, faster time-to-value",
            "Cons: Vendor lock-in, cost at scale ($$$)",
            "When: Medium org, needs streaming, willing to pay for managed",
        ],
    },
    "Online Store Technology": {
        "Redis": [
            "Latency: <1ms, great for point lookups",
            "Cons: Memory-only (expensive at scale), no complex queries",
            "Best for: High-frequency features, simple key-value",
        ],
        "DynamoDB": [
            "Latency: 1-5ms, auto-scaling",
            "Cons: AWS lock-in, cost unpredictable at high QPS",
            "Best for: Variable traffic, AWS ecosystem",
        ],
        "Cassandra": [
            "Latency: 2-10ms, wide-column model",
            "Pros: Good for entity+feature_name keys, scales well",
            "Best for: Multi-dimensional feature lookups, cost-sensitive",
        ],
    },
    "Common Failure Modes": [
        "Materialization lag: Batch features are hours stale during pipeline delays",
        "Schema drift: Feature definition changes break downstream consumers",
        "Cold start: New entities have no features (use default values or content features)",
        "Online/offline store sync failures: Online store has stale data, model gets wrong values",
        "Feature explosion: Teams add features without governance → 10,000 unused features",
        "Skew from on-demand features: Computed differently in training vs serving codepaths",
    ]
}

for category, content in tradeoffs.items():
    print(f"\n{'='*65}")
    print(f"  {category}")
    print(f"{'='*65}")
    if isinstance(content, dict):
        for subcategory, points in content.items():
            print(f"\n  [{subcategory}]")
            for p in points:
                print(f"    • {p}")
    elif isinstance(content, list):
        for p in content:
            print(f"  ⚠ {p}")

## 6. Feature Registry and Governance

At scale, discoverability and governance become critical. Without a feature registry, teams duplicate feature work and features become stale zombies consuming compute. The registry is the "API documentation" for features.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import datetime

@dataclass
class FeatureDefinition:
    """A feature definition in the feature registry."""
    name: str
    entity: str                    # e.g., 'user', 'item', 'user_item'
    dtype: str                     # e.g., 'float32', 'int64', 'embedding_256'
    description: str
    owner_team: str
    computation_type: str          # 'batch', 'streaming', 'on_demand'
    update_frequency_hours: float
    tags: List[str] = field(default_factory=list)
    upstream_sources: List[str] = field(default_factory=list)
    downstream_models: List[str] = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat()[:10])
    
    def to_dict(self):
        return self.__dict__


class FeatureRegistry:
    """In-memory feature registry (production: backed by DB + search index)."""
    
    def __init__(self):
        self._features: dict = {}
    
    def register(self, feature: FeatureDefinition):
        self._features[feature.name] = feature
        print(f"  Registered: {feature.name} [{feature.entity}] ({feature.computation_type})")
    
    def get(self, name: str) -> Optional[FeatureDefinition]:
        return self._features.get(name)
    
    def search_by_entity(self, entity: str) -> List[FeatureDefinition]:
        return [f for f in self._features.values() if f.entity == entity]
    
    def search_by_tag(self, tag: str) -> List[FeatureDefinition]:
        return [f for f in self._features.values() if tag in f.tags]
    
    def get_feature_lineage(self, model_name: str) -> List[str]:
        """Find all features used by a given model."""
        return [name for name, f in self._features.items() if model_name in f.downstream_models]
    
    def impact_analysis(self, source: str) -> List[str]:
        """Which features would be affected if this source changes?"""
        return [name for name, f in self._features.items() if source in f.upstream_sources]
    
    def summary(self):
        print(f"\nRegistry Summary: {len(self._features)} features registered")
        by_type = {}
        for f in self._features.values():
            by_type[f.computation_type] = by_type.get(f.computation_type, 0) + 1
        for t, count in sorted(by_type.items()):
            print(f"  {t}: {count} features")


# Build a sample registry
registry = FeatureRegistry()

features = [
    FeatureDefinition('user_age_days', 'user', 'int64', 'Days since account creation',
                      'growth_team', 'batch', 24, ['demographic'], ['user_table'],
                      ['ranking_model', 'churn_model']),
    FeatureDefinition('user_7d_ctr', 'user', 'float32', '7-day click-through rate across all content',
                      'ads_team', 'batch', 24, ['engagement', 'historical'], ['click_log'],
                      ['ranking_model', 'ads_ranker']),
    FeatureDefinition('user_1h_session_count', 'user', 'int64', 'Number of sessions in last 1 hour',
                      'ranking_team', 'streaming', 0.17, ['realtime', 'engagement'], ['kafka_events'],
                      ['ranking_model']),
    FeatureDefinition('item_embedding_256', 'item', 'embedding_256', '256-dim content embedding from two-tower model',
                      'ranking_team', 'batch', 48, ['embedding', 'content'], ['item_text', 'item_image'],
                      ['retrieval_model', 'ranking_model']),
    FeatureDefinition('item_7d_click_rate', 'item', 'float32', 'Item click rate over 7 days',
                      'content_team', 'batch', 24, ['engagement', 'historical'], ['click_log'],
                      ['ranking_model']),
    FeatureDefinition('query_item_similarity', 'user_item', 'float32', 'Dot product of query and item embeddings at request time',
                      'ranking_team', 'on_demand', 0, ['similarity', 'realtime'], [],
                      ['ranking_model']),
]

print("Registering features:")
for f in features:
    registry.register(f)

registry.summary()

print("\n" + "="*55)
print("Features used by ranking_model:")
for fname in registry.get_feature_lineage('ranking_model'):
    f = registry.get(fname)
    print(f"  • {fname} ({f.computation_type}, {f.update_frequency_hours:.1f}h staleness)")

print("\nImpact if click_log source changes:")
for fname in registry.impact_analysis('click_log'):
    print(f"  ⚠ {fname}")

## Summary

| Concept | Key Point | Interview Signal |
|---------|-----------|------------------|
| Why feature stores exist | Eliminate training-serving skew; one definition | Can explain the core problem without prompting |
| Offline vs online store | Offline: historical+PIT joins; Online: low-latency lookup | Know the latency difference (seconds vs ms) |
| Point-in-time joins | Use feature values *at event time*, not current | Critical for avoiding data leakage |
| Batch vs streaming vs on-demand | Freshness-cost tradeoff; most features are batch | Know when to use each type |
| Online store choice | Redis (<1ms), DynamoDB (1-5ms), Cassandra (2-10ms) | Justify choice based on latency/cost |
| Feature registry | Discoverability, lineage, impact analysis | Shows awareness of operational maturity |
| When NOT to use | Small team, simple models, low QPS | Judgment that you don't over-engineer |
| Key failure modes | Materialization lag, cold start, schema drift | Shows production ML experience |